# Trace-Based Imitation

**Last updated:** 2026-04-09 12:00 PT

**Track:** Learning | **Sub-Ability:** Observational learning

Can the model learn a novel procedure by observing worked examples?
Pre/post learning framework: observe execution traces, then apply to new inputs.

**Difficulty grid:** procedure complexity x num traces x 3 seeds = 27 items

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import random
import re
import sqlite3
import json
import time
import urllib.request
import urllib.error

print('Available models:', list(kbench.llms.keys()))
print('Default model:', kbench.llm)

In [ ]:
def strip_thinking(text: str) -> str:
    """Remove <think>...</think> blocks from reasoning model output."""
    if '</think>' in text:
        return text.split('</think>', 1)[1].strip()
    return text.strip()

def extract_short_answer(response: str) -> str:
    """Extract a short answer (last short line, stripped of formatting)."""
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    if not lines: return ''
    for line in reversed(lines):
        clean = line.strip('"\'\'\u201c\u201d').rstrip('.!?')
        if len(clean) <= 30:
            return clean.lower().strip()
    return lines[-1].strip('"\'\'\u201c\u201d').rstrip('.!?').lower().strip()

def extract_number(response: str) -> str:
    """Extract a number from model response."""
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    for line in reversed(lines):
        clean = line.strip('.!?, ')
        if re.match(r'^-?\d+$', clean): return clean
    nums = re.findall(r'-?\d+', text)
    return nums[-1] if nums else ''

In [ ]:
# Constants used in results analysis
COMPLEXITY_LEVELS = ['simple', 'medium', 'complex']
TRACE_COUNTS = [2, 3, 4]
SEEDS = 3

In [ ]:
# === Load dataset from Kaggle ===
dataset = pd.read_csv('/kaggle/input/trace-based-imitation-benchmark/trace_based_imitation_dataset.csv')
print(f'Dataset: {len(dataset)} items')
print(dataset[['task_id', 'difficulty_label', 'procedure_name', 'test_input', 'expected']].to_string(index=False))

In [ ]:
DB_PATH = 'trace_imitation_runs.db'
db = sqlite3.connect(DB_PATH)
db.execute('DROP TABLE IF EXISTS runs')
db.execute('''
    CREATE TABLE runs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT, model TEXT, task_id TEXT,
        complexity TEXT, n_traces INTEGER, difficulty_label TEXT,
        seed INTEGER, procedure_name TEXT, item_desc TEXT,
        test_input TEXT, expected TEXT,
        pre_raw TEXT, pre_extracted TEXT, pre_correct INTEGER,
        study_notes TEXT,
        post_raw TEXT, post_extracted TEXT, post_correct INTEGER,
        pre_time_s REAL, study_time_s REAL, post_time_s REAL
    )
''')
db.commit()
print(f'SQLite ready (fresh): {DB_PATH}')

In [ ]:
# === Prompt templates ===

PRE_P = """You are shown worked examples of a novel procedure applied to lists of numbers.
Each example shows the input, intermediate steps, and final output.

{material}

Apply the same procedure to this input: {test_input}

Show your work step by step, then give the final answer as a single number on the last line."""

STUDY_P = """You are shown worked examples of a novel procedure applied to lists of numbers.
Each example shows the input, intermediate steps, and final output.

{material}

Your task:
1. Analyze these examples step by step.
2. Write down the EXACT procedure being followed.
3. Be precise about the order of operations, any conditions or branching rules, and how each step transforms the data.
4. Verify your understanding by mentally re-running the procedure on at least one example.

Write clear, detailed notes that would let someone else reproduce this procedure exactly."""

POST_P = """You previously studied a procedure from worked examples and wrote these notes:

--- YOUR NOTES ---
{notes}
--- END NOTES ---

Here are the original worked examples for reference:
{material}

Apply the procedure to this new input: {test_input}

Show your work step by step, then give the final answer as a single number on the last line."""

## Evaluation

In [ ]:
available = list(kbench.llms.keys())
@kbench.task(name='trace_based_imitation',
             description='Pre/post learning: imitate novel procedures from worked traces')
def trace_based_imitation(llm, material: str, test_input: str, expected: str,
                          complexity: str, n_traces: int, difficulty_label: str,
                          seed: int, procedure_name: str, item_desc: str,
                          procedure_desc: str, expected_steps: str) -> dict:
    model_name = str(llm)
    ts = time.strftime('%Y-%m-%d %H:%M:%S')
    task_id = f'{complexity}_{n_traces}traces_{seed}'

    t0 = time.time()
    pre_raw = llm.prompt(PRE_P.format(material=material, test_input=test_input))
    pre_time = time.time() - t0
    pre_extracted = extract_number(pre_raw)
    pre_correct = pre_extracted == str(expected)

    t0 = time.time()
    study_raw = llm.prompt(STUDY_P.format(material=material))
    study_time = time.time() - t0
    notes = strip_thinking(study_raw)

    t0 = time.time()
    post_raw = llm.prompt(POST_P.format(notes=notes, material=material, test_input=test_input))
    post_time = time.time() - t0
    post_extracted = extract_number(post_raw)
    post_correct = post_extracted == str(expected)

    try:
        db.execute(
            """INSERT INTO runs (timestamp,model,task_id,complexity,n_traces,difficulty_label,
               seed,procedure_name,item_desc,test_input,expected,
               pre_raw,pre_extracted,pre_correct,
               study_notes,post_raw,post_extracted,post_correct,
               pre_time_s,study_time_s,post_time_s)
               VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)""",
            (ts, model_name, task_id, complexity, int(n_traces), difficulty_label,
             int(seed), procedure_name, item_desc, test_input, str(expected),
             pre_raw, pre_extracted, int(pre_correct),
             notes, post_raw, post_extracted, int(post_correct),
             pre_time, study_time, post_time))
        db.commit()
    except Exception as e:
        print(f'  [SQLite warn] {e}')

    b = 'Y' if pre_correct else 'N'
    l = 'Y' if post_correct else 'N'
    print(f'[{model_name:40s}] [{difficulty_label:18s}] {procedure_name:15s} expected={expected:>6s}  '
          f'pre="{pre_extracted}"({b})  post="{post_extracted}"({l})')
    return {'pre_correct': pre_correct, 'post_correct': post_correct}

try:
    runs = trace_based_imitation.evaluate(llm=[kbench.llm], evaluation_data=dataset.set_index('task_id'),
                                      n_jobs=1, timeout=3600, max_attempts=1)
    print(f'\nDone: {len(runs.as_dataframe())} items')
except Exception as e:
    print(f'SDK post-processing error (non-fatal): {e}')

## Results

In [ ]:
results = pd.read_sql('SELECT * FROM runs ORDER BY model, task_id', db)
print(f'Total runs: {len(results)}\n')

# Per-model summary
print('SCALING CHECK — Pre-learning accuracy by model:')
print('=' * 70)
for model in sorted(results['model'].unique()):
    sub = results[results['model'] == model]
    pre = sub['pre_correct'].mean()
    post = sub['post_correct'].mean()
    gain = post - pre
    print(f'  {str(model):45s}  pre={pre:.1%}  post={post:.1%}  gain={gain:+.1%}  (n={len(sub)})')

# Per model x complexity
print()
print('By model x complexity (PRE):')
print('-' * 70)
for model in sorted(results['model'].unique()):
    row = f'  {str(model):45s}'
    for c in COMPLEXITY_LEVELS:
        sub = results[(results['model'] == model) & (results['complexity'] == c)]
        if len(sub) > 0:
            row += f'  {c[:4]}={sub["pre_correct"].mean():.0%}'
    print(row)

# Per model x n_traces
print()
print('By model x traces (PRE):')
print('-' * 70)
for model in sorted(results['model'].unique()):
    row = f'  {str(model):45s}'
    for nt in TRACE_COUNTS:
        sub = results[(results['model'] == model) & (results['n_traces'] == nt)]
        if len(sub) > 0:
            row += f'  {nt}tr={sub["pre_correct"].mean():.0%}'
    print(row)

print()
print(f'Timing: pre={results["pre_time_s"].mean():.1f}s  study={results["study_time_s"].mean():.1f}s  post={results["post_time_s"].mean():.1f}s')

In [ ]:
print('STUDY NOTES INSPECTION')
print('=' * 60)
for _, row in results.iterrows():
    b = 'Y' if row['pre_correct'] else 'N'
    l = 'Y' if row['post_correct'] else 'N'
    print(f'\n--- [{row["difficulty_label"]}] seed={row["seed"]} ---')
    print(f'Procedure: {row["procedure_name"]} | Item: {row["item_desc"]}')
    print(f'Expected: {row["expected"]}')
    print(f'Pre: "{row["pre_extracted"]}" ({b})  Post: "{row["post_extracted"]}" ({l})')
    print(f'Notes (first 300 chars): {row["study_notes"][:300]}')
    print('...' if len(str(row['study_notes'])) > 300 else '')

In [ ]:
!pip install -q matplotlib
import matplotlib.pyplot as plt

labels = sorted(results['difficulty_label'].unique())
pre_scores = [results[results['difficulty_label']==l]['pre_correct'].mean() for l in labels]
post_scores = [results[results['difficulty_label']==l]['post_correct'].mean() for l in labels]

x = range(len(labels))
width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax1 = axes[0]
ax1.bar([i-width/2 for i in x], pre_scores, width, label='Pre-learning', color='#4C72B0')
ax1.bar([i+width/2 for i in x], post_scores, width, label='Post-learning', color='#55A868')
ax1.set_ylabel('Accuracy')
ax1.set_title('Trace-Based Imitation: Pre vs Post-Learning')
ax1.set_xticks(list(x))
ax1.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax1.legend()
ax1.set_ylim(0, 1.05)

ax2 = axes[1]
gains = [p-b for b,p in zip(pre_scores, post_scores)]
colors = ['#55A868' if g>=0 else '#C44E52' for g in gains]
ax2.bar(range(len(labels)), gains, color=colors)
ax2.set_ylabel('Learning Gain')
ax2.set_title('Learning Gain by Difficulty')
ax2.set_xticks(list(x))
ax2.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.savefig('trace_imitation_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Post-run: upload to BigQuery + Google Sheets via REST API ===
# Safe to do after benchmark — SDK can break, doesn't matter.

results_upload = pd.read_sql('SELECT * FROM runs', db)
db.close()
print(f'SQLite closed. {len(results_upload)} rows to upload.\n')

auth_token = None
gcp_project = None
try:
    import google.auth
    import google.auth.transport.requests
    creds, project = google.auth.default()
    creds.refresh(google.auth.transport.requests.Request())
    auth_token = creds.token
    gcp_project = project
    print(f'Authenticated. Project: {gcp_project}')
except Exception as e:
    print(f'Google auth not available: {e}')

BQ_DATASET = 'benchmark_runs'

# --- BigQuery ---
if auth_token and gcp_project:
    BQ_TABLE = DB_PATH.replace('_runs.db', '')
    try:
        import urllib.parse
        ds_url = f'https://api.kaggle.com/api/v1/bigquery/projects/{gcp_project}/datasets'
        ds_body = json.dumps({'datasetReference': {'datasetId': BQ_DATASET}, 'location': 'US'}).encode()
        req = urllib.request.Request(ds_url, data=ds_body, method='POST',
                                     headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
        try: urllib.request.urlopen(req)
        except urllib.error.HTTPError as e:
            if e.code != 409: raise

        schema_fields = [{'name': c, 'type': 'STRING' if results_upload[c].dtype == 'object' else
                          'INTEGER' if 'correct' in c or c in ('seed','id','n_traces') else 'FLOAT'}
                         for c in results_upload.columns]
        create_url = f'https://bigquery.googleapis.com/bigquery/v2/projects/{gcp_project}/datasets/{BQ_DATASET}/tables'
        create_body = json.dumps({'tableReference': {'projectId': gcp_project, 'datasetId': BQ_DATASET, 'tableId': BQ_TABLE},
                                  'schema': {'fields': schema_fields}}).encode()
        req = urllib.request.Request(create_url, data=create_body, method='POST',
                                     headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
        try: urllib.request.urlopen(req)
        except urllib.error.HTTPError as e:
            if e.code != 409: raise

        table_url = f'https://bigquery.googleapis.com/bigquery/v2/projects/{gcp_project}/datasets/{BQ_DATASET}/tables/{BQ_TABLE}/insertAll'
        rows_json = [{'json': {c: str(v) if pd.notna(v) else '' for c, v in row.items()}} for _, row in results_upload.iterrows()]
        insert_body = json.dumps({'rows': rows_json}).encode()
        req = urllib.request.Request(table_url, data=insert_body, method='POST',
                                     headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
        urllib.request.urlopen(req)
        print(f'BigQuery: {len(results_upload)} rows -> {gcp_project}.{BQ_DATASET}.{BQ_TABLE}')
    except Exception as e:
        print(f'BigQuery failed (non-fatal): {e}')

# --- Google Sheets ---
if auth_token:
    SHEET_NAME = f'Learning Bench — {BQ_TABLE} Runs'
    try:
        import urllib.parse
        search_url = ('https://www.googleapis.com/drive/v3/files'
                      f'?q=name%3D%27{urllib.parse.quote(SHEET_NAME)}%27+and+mimeType%3D%27application/vnd.google-apps.spreadsheet%27'
                      '&fields=files(id,webViewLink)')
        req = urllib.request.Request(search_url, headers={'Authorization': f'Bearer {auth_token}'})
        resp = json.loads(urllib.request.urlopen(req).read())
        files = resp.get('files', [])
        if files:
            sid = files[0]['id']
        else:
            create_url = 'https://sheets.googleapis.com/v4/spreadsheets'
            req = urllib.request.Request(create_url, data=json.dumps({'properties': {'title': SHEET_NAME}}).encode(),
                                         method='POST', headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
            resp = json.loads(urllib.request.urlopen(req).read())
            sid = resp['spreadsheetId']
            header = list(results_upload.columns)
            body = json.dumps({'values': [header]}).encode()
            req = urllib.request.Request(f'https://sheets.googleapis.com/v4/spreadsheets/{sid}/values/Sheet1!A1:append?valueInputOption=RAW',
                                         data=body, method='POST', headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
            urllib.request.urlopen(req)

        data_rows = [[str(v) if pd.notna(v) else '' for v in row] for _, row in results_upload.iterrows()]
        body = json.dumps({'values': data_rows}).encode()
        req = urllib.request.Request(f'https://sheets.googleapis.com/v4/spreadsheets/{sid}/values/Sheet1!A1:append?valueInputOption=RAW',
                                     data=body, method='POST', headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
        urllib.request.urlopen(req)
        print(f'Sheets: {len(results_upload)} rows -> "{SHEET_NAME}"')
    except Exception as e:
        print(f'Sheets failed (non-fatal): {e}')

print(f'\nDone. SQLite: {DB_PATH}')